<h4>This notebook will load a dataset that has finance news reports and we will convert them into chunks -> embed the chunks-> load into a vector database</h4>

In [2]:
# !pip install pinecone
# !pip install openai
# !pip install langchain
# !pip install kagglehub
# !pip install -U langchain-community
# %pip install rapidfuzz
# %pip install google-generativeai
%pip install dateparser

  Attempting uninstall: pytz
    Found existing installation: pytz 2024.1
    Uninstalling pytz-2024.1:
      Successfully uninstalled pytz-2024.1
Note: you may need to restart the kernel to use updated packages.


In [3]:
import kagglehub
import os
import pinecone
from openai import OpenAI
import openai
import pandas as pd
import numpy as np
import langchain
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import Pinecone
from rapidfuzz import process
import google.generativeai as genai
import json
from datetime import datetime
from dotenv import load_dotenv
from pinecone import Pinecone
import dateparser

In [59]:
load_dotenv()

True

In [ ]:


# Download latest version
path = kagglehub.dataset_download("notlucasp/financial-news-headlines")

print("Path to dataset files:", path)

100%|██████████| 3.91M/3.91M [00:01<00:00, 2.45MB/s]


Extracting files...
Path to dataset files: C:\Users\cyber\.cache\kagglehub\datasets\notlucasp\financial-news-headlines\versions\2


In [5]:
path = "C:\\Users\\cyber\\.cache\\kagglehub\\datasets\\notlucasp\\financial-news-headlines\\versions\\2"

In [6]:
df_cnbc = pd.read_csv(path + "/cnbc_headlines.csv")
df_gaurdian = pd.read_csv(path + "/guardian_headlines.csv")
df_reuters = pd.read_csv(path + "/reuters_headlines.csv")


In [7]:
df_symbols_valid = pd.read_csv("C:/Users/cyber/.cache/kagglehub/datasets/jacksoncrow/stock-market-dataset/versions/2/symbols_valid_meta.csv")

In [8]:
print(df_cnbc.describe())
print(df_gaurdian.describe())
print(df_reuters.describe())

                                                Headlines  \
count                                                2800   
unique                                               2788   
top     Cramer: I helped investors through the 2010 fl...   
freq                                                    2   

                                 Time  \
count                            2800   
unique                           2474   
top      8:11  PM ET Fri,  2 Nov 2018   
freq                                6   

                                              Description  
count                                                2800  
unique                                               2618  
top     "Mad Money" host Jim Cramer rings the lightnin...  
freq                                                  147  
             Time                                          Headlines
count       17800                                              17800
unique        774                                 

<h4>So the number of entries from all the sources are decent enough to be stored in a database</h4>

In [9]:
df_gaurdian.head()


,Time,Headlines
0,18-Jul-20,Johnson is asking Santa for a Christmas recovery
1,18-Jul-20,‘I now fear the worst’: four grim tales of wor...
2,18-Jul-20,Five key areas Sunak must tackle to serve up e...
3,18-Jul-20,Covid-19 leaves firms ‘fatally ill-prepared’ f...
4,18-Jul-20,The Week in Patriarchy \n\n\n Bacardi's 'lad...


In [10]:
df_cnbc.head()

,Headlines,Time,Description
0,Jim Cramer: A better way to invest in the Covi...,"7:51 PM ET Fri, 17 July 2020","""Mad Money"" host Jim Cramer recommended buying..."
1,Cramer's lightning round: I would own Teradyne,"7:33 PM ET Fri, 17 July 2020","""Mad Money"" host Jim Cramer rings the lightnin..."
2,NaN,NaN,NaN
3,"Cramer's week ahead: Big week for earnings, ev...","7:25 PM ET Fri, 17 July 2020","""We'll pay more for the earnings of the non-Co..."
4,IQ Capital CEO Keith Bliss says tech and healt...,"4:24 PM ET Fri, 17 July 2020","Keith Bliss, IQ Capital CEO, joins ""Closing Be..."


In [11]:
df_reuters.head(5)

,Headlines,Time,Description
0,TikTok considers London and other locations fo...,Jul 18 2020,TikTok has been in discussions with the UK gov...
1,Disney cuts ad spending on Facebook amid growi...,Jul 18 2020,Walt Disney has become the latest company to ...
2,Trail of missing Wirecard executive leads to B...,Jul 18 2020,Former Wirecard chief operating officer Jan M...
3,Twitter says attackers downloaded data from up...,Jul 18 2020,Twitter Inc said on Saturday that hackers were...
4,U.S. Republicans seek liability protections as...,Jul 17 2020,A battle in the U.S. Congress over a new coron...


In [12]:
print("cnbc", df_cnbc['Time'].head(), sep="\n", end="\n\n")
print("guardian", df_gaurdian['Time'].head(), sep="\n", end="\n\n")
print("reuters", df_reuters['Time'].head(), sep="\n", end="\n\n")


cnbc
0     7:51  PM ET Fri, 17 July 2020
1     7:33  PM ET Fri, 17 July 2020
2                               NaN
3     7:25  PM ET Fri, 17 July 2020
4     4:24  PM ET Fri, 17 July 2020
Name: Time, dtype: object

guardian
0    18-Jul-20
1    18-Jul-20
2    18-Jul-20
3    18-Jul-20
4    18-Jul-20
Name: Time, dtype: object

reuters
0    Jul 18 2020
1    Jul 18 2020
2    Jul 18 2020
3    Jul 18 2020
4    Jul 17 2020
Name: Time, dtype: object



<h4>So the date in the cnbc needs to be formatted to just the date as the time doesn't matter, guardian is in a different format so that's needs to be changed as well only reuters is in the correct format

In [13]:
df_reuters = df_reuters.rename(columns={"Time": "Date"})  
#convert to datetime
df_reuters['Date'] = pd.to_datetime(df_reuters['Date'])
df_reuters['Date'] = df_reuters['Date'].dt.strftime('%Y-%m-%d')
df_reuters['Date'].head(5) #check if the conversion was successful

0    2020-07-18
1    2020-07-18
2    2020-07-18
3    2020-07-18
4    2020-07-17
Name: Date, dtype: object

In [14]:
df_reuters.isnull().sum() #check for null values

Headlines      0
Date           0
Description    0
dtype: int64

So only the first 3 letters of every month, convert every month into numbers then convert to dateTime

In [15]:
df_gaurdian.dropna(how='any', inplace=True) #drop null values

In [16]:
df_gaurdian = df_gaurdian.rename(columns={"Time": "Date"})  

In [17]:
df_gaurdian[df_gaurdian['Date'] == '18']

,Date,Headlines


In [18]:
months = [
    'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
    'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec'
]
new = {
    'fine' : [],
    'wrong' : []
}
df_gaurdian_date_fixed = df_gaurdian['Date'].str.split('-').apply(lambda x: f"1-{x[0]}-{x[1]}" if  x[1] not in months else f"{x[0]}-{x[1]}-{x[2]}")
print(new['wrong'], len(new['wrong']), sep="\n")

[]
0


In [19]:
df_gaurdian_date_fixed.head()

0    18-Jul-20
1    18-Jul-20
2    18-Jul-20
3    18-Jul-20
4    18-Jul-20
Name: Date, dtype: object

In [20]:
months = {
    'Jan': '01',
    'Feb': '02',
    'Mar': '03',
    'Apr': '04',
    'May': '05',
    'Jun': '06',
    'Jul': '07',
    'Aug': '08',
    'Sep': '09',
    'Oct': '10',
    'Nov': '11',
    'Dec': '12'
}

df_gaurdian_temp = df_gaurdian_date_fixed.str.split('-').apply(lambda x:f"20{x[2]}-{months[x[1]]}-{x[0]}" )

df_gaurdian_temp.head(5) #check if the conversion was successful

0    2020-07-18
1    2020-07-18
2    2020-07-18
3    2020-07-18
4    2020-07-18
Name: Date, dtype: object

In [21]:
df_gaurdian['Date'] = df_gaurdian_temp

In [22]:
df_gaurdian.head()

,Date,Headlines
0,2020-07-18,Johnson is asking Santa for a Christmas recovery
1,2020-07-18,‘I now fear the worst’: four grim tales of wor...
2,2020-07-18,Five key areas Sunak must tackle to serve up e...
3,2020-07-18,Covid-19 leaves firms ‘fatally ill-prepared’ f...
4,2020-07-18,The Week in Patriarchy \n\n\n Bacardi's 'lad...


In [23]:
df_cnbc['Time'].describe()
df_cnbc['Time'].isnull().sum()
#Get rows where time is null and headlines is not null
df_cnbc[df_cnbc['Time'].isnull() & df_cnbc['Headlines'].notnull()]
#drop these rows
df_cnbc.dropna(subset=['Time'], inplace=True) #drop null values
df_cnbc['Time'].isnull().sum() #check if the conversion was successful

0

In [24]:
df_cnbc_temp = df_cnbc['Time'].str.split(',').apply(lambda x: x[1].lstrip() )
# df_cnbc_temp = df_cnbc_temp.str.split(',').apply(lambda x:f"20{x[2]}-{months[x[1]]}-{x[0]}")
df_cnbc_temp.head(5)

0    17 July 2020
1    17 July 2020
3    17 July 2020
4    17 July 2020
5    16 July 2020
Name: Time, dtype: object

In [25]:
months = [
    'Jan', 'Feb', 'March', 'April', 'May', 'June',
    'July', 'Aug', 'Sept', 'Oct', 'Nov', 'Dec'
]
new = {
    'fine' : [],
    'wrong' : []
}
df_cnbc_date_fixed = df_cnbc_temp.str.split(' ').apply(lambda x: f"{x[2]}-{x[1]}-{x[0]}")
df_cnbc_date_fixed.head(5) #check if the conversion was successful

0    2020-July-17
1    2020-July-17
3    2020-July-17
4    2020-July-17
5    2020-July-16
Name: Time, dtype: object

In [26]:
print(new['fine'][:5], len(new['fine']), sep="\n", end="\n\n")

[]
0



In [27]:
months = {
    'Jan': '01',
    'Feb': '02',
    'March': '03',
    'April': '04',
    'May': '05',
    'June': '06',
    'July': '07',
    'Aug': '08',
    'Sept': '09',
    'Oct': '10',
    'Nov': '11',
    'Dec': '12'
}

df_cnbc_temp = df_cnbc_date_fixed.str.split('-').apply(lambda x:f"{x[0]}-{months[x[1]]}-{x[2]}" )

df_cnbc_temp.head(5) #check if the conversion was successful

0    2020-07-17
1    2020-07-17
3    2020-07-17
4    2020-07-17
5    2020-07-16
Name: Time, dtype: object

In [28]:
df_cnbc['Date'] = df_cnbc_temp
df_cnbc.drop(columns=['Time'], inplace=True) #drop the old time column


In [29]:
#convert to datetime
df_cnbc['Date'] = pd.to_datetime(df_cnbc['Date'])
df_cnbc['Date'] = df_cnbc['Date'].dt.strftime('%Y-%m-%d')
df_cnbc['Date'].head(5) #check if the conversion was successful

0    2020-07-17
1    2020-07-17
3    2020-07-17
4    2020-07-17
5    2020-07-16
Name: Date, dtype: object

In [30]:
df_gaurdian['Date'] = pd.to_datetime(df_gaurdian['Date'])
df_gaurdian['Date'] = df_gaurdian['Date'].dt.strftime('%Y-%m-%d')

In [31]:
df_cnbc.sort_values(by='Date', inplace=True)
df_gaurdian.sort_values(by='Date', inplace=True)    
df_reuters.sort_values(by='Date', inplace=True)

In [32]:
df_reuters['Date'].head(5)

32769    2018-03-20
32738    2018-03-20
32737    2018-03-20
32736    2018-03-20
32735    2018-03-20
Name: Date, dtype: object

<h4> We need to convert the news information into chunks and store them, so we need to identify the sentiment in the data </h4>

In [40]:
df_reuters['Description'][1]

'Walt Disney  has become the latest company to slash its advertising spending on Facebook Inc  as the social media giant faces an ad boycott over its handling of hate speech and controversial content, the Wall Street Journal reported on Saturday, citing people familiar with the situation.'

In [41]:
df_reuters.head()

,Headlines,Date,Description
32769,UK will always consider ways to improve data l...,2018-03-20,Britain will consider any suggestions to give ...
32738,Senate Democrat wants Facebook CEO Zuckerberg ...,2018-03-20,"U.S. Senator Dianne Feinstein, the top Democra..."
32737,"Factbox: How United States, others regulate au...",2018-03-20,An Uber self-driving sport utility vehicle str...
32736,Cambridge Analytica played key Trump campaign ...,2018-03-20,The suspended chief executive of UK-based poli...
32735,Start of AT&T-Time Warner trial delayed until ...,2018-03-20,Opening statements in the trial to decide if A...


In [43]:
from tqdm import tqdm
for i in tqdm(range(len(df_reuters))):
    df_reuters['Description'][i] = df_reuters['Description'][i].replace('\n', ' ')
df_reuters.head()
df_reuters['Description'][1]



  0%|          | 0/32770 [00:00<?, ?it/s]

100%|██████████| 32770/32770 [00:21<00:00, 1539.13it/s]


'Walt Disney  has become the latest company to slash its advertising spending on Facebook Inc  as the social media giant faces an ad boycott over its handling of hate speech and controversial content, the Wall Street Journal reported on Saturday, citing people familiar with the situation.'

In [44]:
df_reuters = df_reuters[((df_reuters['Date'] >= '2015-01-01') & (df_reuters['Date'] <= '2020-01-01'))]
df_reuters


,Headlines,Date,Description
32769,UK will always consider ways to improve data l...,2018-03-20,Britain will consider any suggestions to give ...
32738,Senate Democrat wants Facebook CEO Zuckerberg ...,2018-03-20,"U.S. Senator Dianne Feinstein, the top Democra..."
32737,"Factbox: How United States, others regulate au...",2018-03-20,An Uber self-driving sport utility vehicle str...
32736,Cambridge Analytica played key Trump campaign ...,2018-03-20,The suspended chief executive of UK-based poli...
32735,Start of AT&T-Time Warner trial delayed until ...,2018-03-20,Opening statements in the trial to decide if A...
...,...,...,...
9210,"Surveillance in a leafy enclave, Ghosn's Tokyo...",2020-01-01,The imposing home where Carlos Ghosn lived for...
9211,Ghosn flight prompts talk of more curbs in Jap...,2020-01-01,"Carlos Ghosn's daring flight from Japan, where..."
9212,"Surveillance in a leafy enclave, Ghosn's Tokyo...",2020-01-01,The imposing home where Carlos Ghosn lived for...
9213,Samsung Electronics chip output at South Korea...,2020-01-01,Samsung Electronics partly halted some semico...


In [45]:
df_cnbc.columns, df_gaurdian.columns, df_reuters.columns

(Index(['Headlines', 'Description', 'Date'], dtype='object'),
 Index(['Date', 'Headlines'], dtype='object'),
 Index(['Headlines', 'Date', 'Description'], dtype='object'))

In [46]:
df_cnbc = df_cnbc[((df_cnbc['Date'] >= '2015-01-01') & (df_cnbc['Date'] <= '2020-01-01'))]
df_cnbc

,Headlines,Description,Date
3079,Cramer: Never buy a stock all at once — you'll...,Jim Cramer doubled down on his key investing r...,2017-12-22
3078,Cramer: I helped investors through the 2010 fl...,"Jim Cramer built on his ""nobody ever made a di...",2017-12-22
3077,Cramer says owning too many stocks and too lit...,Jim Cramer broke down why owning fewer stocks ...,2017-12-22
3075,Markets lack Christmas cheer,"According to Kensho, here's how markets have f...",2017-12-26
3074,S&P tends to start new year bullish after this...,The S&P is on track to end the year up 20 perc...,2017-12-27
...,...,...,...
638,"S&P 500, Russell 2000 and Nasdaq are at risk o...","The S&P 500, Russell 2000 and Nasdaq Composite...",2019-12-30
633,"Cramer Remix: Here's where your first $10,000 ...",Jim Cramer shares his advice for how investors...,2019-12-31
632,Cramer explains the magic of compounding,Jim Cramer explains why young investors stand ...,2019-12-31
631,Cramer makes the case for including bonds in y...,Jim Cramer addresses the lingering question of...,2019-12-31


In [47]:
def analyze_news(headline, description):
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    model = genai.GenerativeModel("gemini-2.0-flash")
    prompt = f"""
    Analyze this financial news article and extract the following information:
    Headline: {headline}
    Description: {description}
    
    Please provide:
    1. List of company ticker symbols mentioned (if any)
    2. Sentiment (positive/negative/neutral)
    3. Industry sector(s) mentioned
    4. Brief summary of the news
    
    Ensure the response is in JSON format.
    Format the response as a JSON object with these keys:
    - tickers: list of ticker symbols
    - sentiment: string
    - sectors: list of sectors
    - summary: string
    """
    try:
        response = client.chat.completions.create(
            model = "gpt-3.5-turbo",
            messages= [{
                "role":"user", 
                "content":prompt
            }],
            temperature= 0,
            response_format = 
            {
                "type":"json_object"
            }
        )
        result = json.loads(response.choices[0].message.content)
        
        # Handle empty response
        if not result:
            return {
                "tickers": [],
                "sentiment": "neutral",
                "sectors": [],
                "summary": ""
            }
        # Try to parse the JSON response
        try:
            # Ensure all required fields exist
            if "tickers" not in result:
                result["tickers"] = []
            if "sentiment" not in result:
                result["sentiment"] = "neutral"
            if "sectors" not in result:
                result["sectors"] = []
            if "summary" not in result:
                result["summary"] = ""
            return result
        except json.JSONDecodeError:
            print(f"Failed to parse JSON response: {result}")
            return {
                "tickers": [],
                "sentiment": "neutral",
                "sectors": [],
                "summary": ""
            }
    
    except Exception as e:
        print(f"Error analyzing news: {e}")
        return {
            "tickers": [],
            "sentiment": "neutral",
            "sectors": [],
            "summary": ""
        }

In [48]:
analyze_news(df_reuters['Headlines'].iloc[0], df_reuters['Description'].iloc[0])

{'tickers': [],
 'sentiment': 'neutral',
 'sectors': [],
 'summary': "The UK will consider suggestions to give the body in charge of upholding data privacy laws more powers, in response to concerns about Facebook's protection of users' data."}

In [51]:

def save_to_json_file(data, file_path):
    if os.path.exists(file_path):
        with open(file_path, "r") as f:
            existing_data = json.load(f)
    else:
        existing_data = []

    existing_data.extend(data)

    try:
        with open(file_path, "w") as f:
            json.dump(existing_data, f, indent=2)
            print(f"Data saved to {file_path}")
    except Exception as e:
        print(f"Error saving file: {e}")

In [52]:
def process_news_data(df, source):
    results = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        try:
            analysis = analyze_news(row['Headlines'], row['Description'])
            
            result = {
                "title": row['Headlines'],
                "date": row['Date'],
                "source": source,
                "metadata": analysis
            }
            if len(results) > 100:
                save_to_json_file(results, f"{source}_news_data.json")
                results = []
            results.append(result)
        except Exception as e:
            print(f"Error processing news data: {e}")
            continue
    return results


In [ ]:
file_path = "all_news.json"
try:
    cnbc_news = process_news_data(df_cnbc, "CNBC")
    
    save_to_json_file(cnbc_news, file_path)
except Exception as e:
    print("Error processing CNBC:", e)

try:
    reuters_news = process_news_data(df_reuters, "Reuters")
    save_to_json_file(reuters_news, file_path)
except Exception as e:
    print("Error processing Reuters:", e)

  5%|▍         | 102/2227 [02:58<1:01:33,  1.74s/it]

Data saved to CNBC_news_data.json


  9%|▉         | 203/2227 [05:59<59:33,  1.77s/it]  

Data saved to CNBC_news_data.json


 14%|█▎        | 304/2227 [09:00<55:26,  1.73s/it]  

...

 30%|███       | 7172/23566 [3:35:33<9:59:13,  2.19s/it] 

Data saved to Reuters_news_data.json


 31%|███       | 7273/23566 [3:38:36<8:22:43,  1.85s/it] 

Data saved to Reuters_news_data.json


 31%|███       | 7303/23566 [3:39:34<8:08:58,  1.80s/it] 

#### This is about 2K headlines from each but there are some where there are no ticker being mentioned or other outliers so we need to remove them.

In [47]:
cnbc_chunk_data  = json.load(open("CNBC_news_data.json", "r"))
cnbc_cleaned_data = []
for i in cnbc_chunk_data:
    if(i['metadata']['tickers'] != []):
        cnbc_cleaned_data.append(i)


        

In [49]:
reuters_chunk_data  = json.load(open("Reuters_news_data.json", "r"))
reuters_cleaned_data = []
for i in reuters_chunk_data:
    if(i['metadata']['tickers'] != []):
        reuters_cleaned_data.append(i)



In [48]:
cnbc_cleaned_data[0]['metadata']['tickers']

['S&P']

In [50]:
len(cnbc_chunk_data), len(reuters_chunk_data), len(cnbc_cleaned_data), len(reuters_cleaned_data)

(2222, 7272, 1226, 3847)

In [51]:
json.dump(cnbc_cleaned_data, open("Cleaned_CNBC_news_data.json", "w"), indent=2)
json.dump(reuters_cleaned_data, open("Cleaned_Reuters_news_data.json", "w"), indent=2)

#### Now that only the cleaned data is available, we will upsert this data into a vector db, for this project it is Pinecone

In [60]:
pc = Pinecone(api_key = os.getenv("PINECONE_API_KEY"))
index = pc.Index(os.getenv("PINECONE_INDEX_NAME"))

In [72]:
client = OpenAI()
text = client.embeddings.create(
  model="text-embedding-3-small",
  input="What was the lowest drop for apple 2019-01-01 to 2020-01-01 ?",
  encoding_format="float"
)
print(text.data[0])

Embedding(embedding=[-0.012552949, 0.00265332, 0.06582674, -0.029961167, 0.02122656, -0.010204608, -0.017944982, 0.040769633, 0.017371621, 0.023056436, 0.03435287, 0.03803702, -0.10071637, -0.024947308, 0.016407887, 0.023080835, 0.017603407, -0.016761662, -0.07421976, 0.010222906, -0.0412576, -0.0031900837, -0.029961167, 0.008441827, 0.06319171, -0.029424405, -0.014797596, -0.029009633, -0.017749796, 0.013931455, 0.038549386, -0.019713862, 0.042087145, -0.009948425, -0.0053127394, -0.01548075, 0.03859818, 0.016322494, -0.017127639, -0.017139837, -0.0022446478, 0.0008890147, -0.0026167226, 0.0015965666, -0.03005876, -0.048577104, 0.004757677, -0.03774424, 0.013431289, 0.031156687, -0.05377395, -0.0027280399, -0.03801262, 0.0072463085, -0.015492949, 0.0051511005, 0.03403569, 0.04028167, 0.0026761934, -0.007404898, 0.0012336413, -0.05411553, 0.027155358, -0.012711538, 0.023654195, -0.004410001, -0.016956849, 0.07768433, 0.041111212, 0.0055079265, -0.0016743364, 0.016712867, -0.019286891, 

In [82]:
from tqdm import tqdm 
client
openai.api_key = os.getenv("OPENAI_API_KEY")
EMBEDDING_MODEL = "text-embedding-3-small"

# Load your news
with open("Cleaned_reuters_news_data.json", "r") as file:
    news = json.load(file)

# Prepare and upload embeddings to Pinecone
batch_size = 100
vectors = []

In [83]:
news[0]

{'title': 'Senate Democrat wants Facebook CEO Zuckerberg to testify',
 'date': '2018-03-20',
 'source': 'Reuters',
 'metadata': {'tickers': ['FB'],
  'sentiment': 'Negative',
  'sectors': ['Technology'],
  'summary': "U.S. Senator Dianne Feinstein is calling for Facebook CEO Mark Zuckerberg to testify in Congress regarding the company's handling of users' data."}}

In [84]:
for i, summary in enumerate(tqdm(news)):

    embedding = client.embeddings.create(
    model="text-embedding-3-small",
    input= summary['title'],
    encoding_format="float"
    ).data[0].embedding

    vectors.append({
        "id": f"article-{i}",
        "values": embedding,
        "metadata": {
            "source": summary['source'],
            "date": summary['date'],
            **summary.get('metadata', {}),  # Unpack metadata if it exists
        }
    })

    # Upload in batches
    if len(vectors) == batch_size or i == len(news) - 1:
        index.upsert(vectors=vectors)
        vectors = []

print("✅ Embeddings uploaded to Pinecone.")

100%|██████████| 3847/3847 [44:35<00:00,  1.44it/s]  

✅ Embeddings uploaded to Pinecone.


#### Now that all the data has been uploaded to the vectordb we can start working on ways to getting user query -> query the vectordb -> ask an LLM 

In [13]:
print(dateparser.parse("quarter 2"))

None
